In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.pricing_feature_combined AS

WITH tour_level_usage AS (
  SELECT 
    tour_id,
    supplier_id,
    
    -- Supplier & Tour Attributes
    MAX(segment) AS supplier_segment,
    MAX(is_managed) AS is_managed,
    MAX(tour_title) AS tour_title,
    MAX(location_name) AS location_name,
    MAX(activity_type) AS activity_type,
    MAX(tour_category) AS tour_category,
    MAX(date_of_creation) AS date_of_creation,
    MAX(date_of_first_online) AS date_of_first_online,
    MAX(is_top_attraction) AS is_top_attraction,
    COUNT(DISTINCT tour_option_id) AS total_options, 
    
    -- ========================================================================
    -- INDIVIDUAL PRICING - ALL TICKET CATEGORIES
    -- ========================================================================
    MAX(has_individual_pricing) AS has_individual_pricing,
    
    -- All unique ticket categories
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'ADULT')) > 0 
      THEN 1 ELSE 0 
    END) AS has_adult_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) IN ('CHILD', 'CHILDREN'))) > 0 
      THEN 1 ELSE 0 
    END) AS has_child_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) IN ('INFANT', 'BABY'))) > 0 
      THEN 1 ELSE 0 
    END) AS has_infant_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'YOUTH')) > 0 
      THEN 1 ELSE 0 
    END) AS has_youth_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'SENIOR')) > 0 
      THEN 1 ELSE 0 
    END) AS has_senior_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'STUDENT')) > 0 
      THEN 1 ELSE 0 
    END) AS has_student_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'MILITARY')) > 0 
      THEN 1 ELSE 0 
    END) AS has_military_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'EU_CITIZEN_STUDENT')) > 0 
      THEN 1 ELSE 0 
    END) AS has_eu_citizen_student_category,
    
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> UPPER(cat.ticketCategory) = 'EU_CITIZEN')) > 0 
      THEN 1 ELSE 0 
    END) AS has_eu_citizen_category,
    
    -- Count of unique individual categories
    MAX(SIZE(individual_categories)) AS num_individual_categories,
    
    -- Total individual pricing tiers (maximum across pricing configs to avoid double counting)
    MAX(AGGREGATE(individual_categories, 0, (acc, cat) -> acc + SIZE(cat.prices))) AS num_individual_tiers,
    
    -- Individual categories with scales (more than 1 price tier)
    MAX(CASE 
      WHEN SIZE(FILTER(individual_categories, cat -> SIZE(cat.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END) AS has_individual_scales,
    
    -- ========================================================================
    -- GROUP PRICING - WITH TIER ANALYSIS
    -- ========================================================================
    MAX(has_group_pricing) AS has_group_pricing,
    
    -- Number of group pricing configurations
    MAX(SIZE(group_categories)) AS num_group_configs,
    
    -- Maximum number of tiers in any group config
    MAX(AGGREGATE(group_categories, 0, (acc, grp) -> 
      CASE WHEN SIZE(grp.prices) > acc THEN SIZE(grp.prices) ELSE acc END
    )) AS max_group_tiers,
    
    -- Total group pricing tiers (maximum across pricing configs to avoid double counting)
    MAX(AGGREGATE(group_categories, 0, (acc, grp) -> acc + SIZE(grp.prices))) AS num_group_tiers,
    
    -- Group with scales (more than 1 tier)
    MAX(CASE 
      WHEN SIZE(FILTER(group_categories, grp -> SIZE(grp.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END) AS has_group_scales,
    
    -- Has additional pax for group
    MAX(CASE 
      WHEN SIZE(FILTER(group_categories, grp -> grp.isAdditionalPaxForGroup = true)) > 0 
      THEN 1 ELSE 0 
    END) AS has_additional_pax_for_group,
    
    -- ========================================================================
    -- ADD-ONS - WITH TIER ANALYSIS
    -- ========================================================================
    MAX(has_addons_configured) AS has_addons,
    
    -- Number of add-ons
    MAX(SIZE(addons)) AS num_addons,
    
    -- Total add-on pricing tiers (maximum across pricing configs to avoid double counting)
    MAX(AGGREGATE(addons, 0, (acc, addon) -> acc + SIZE(addon.prices))) AS num_addon_tiers,
    
    -- Add-ons with scales (more than 1 price tier)
    MAX(CASE 
      WHEN SIZE(FILTER(addons, addon -> SIZE(addon.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END) AS has_addon_scales,
    
    -- ========================================================================
    -- PRICING SCALES - SEPARATED: NATIVE vs API
    -- ========================================================================
    
    -- Native Scales (NOT via API)
    MAX(CASE 
      WHEN has_any_scale_pricing = 1 AND (uses_pricing_scales_over_api IS NULL OR uses_pricing_scales_over_api = FALSE)
      THEN 1 ELSE 0 
    END) AS has_native_scales,
    
    -- API Scales
    MAX(CASE WHEN uses_pricing_scales_over_api = TRUE THEN 1 ELSE 0 END) AS has_api_scales,
    
    -- Total scales (for backward compatibility)
    MAX(has_any_scale_pricing) AS has_scale_pricing,
    
    -- ========================================================================
    -- API PRICING - SEPARATED: STATIC vs LIVE DYNAMIC
    -- ========================================================================
    
    -- Static API Pricing (prices from API but not real-time)
    MAX(CASE 
      WHEN uses_prices_over_api = TRUE AND (uses_live_availability_and_price IS NULL OR uses_live_availability_and_price = FALSE)
      THEN 1 ELSE 0 
    END) AS has_static_api_pricing,
    
    -- Live Dynamic Pricing (real-time pricing and availability)
    MAX(CASE WHEN uses_live_availability_and_price = TRUE THEN 1 ELSE 0 END) AS has_live_dynamic_pricing,
    
    -- Any API pricing (for backward compatibility)
    MAX(CASE WHEN uses_prices_over_api = TRUE THEN 1 ELSE 0 END) AS uses_api_pricing,
    
    -- ========================================================================
    -- OPERATIONAL FEATURES
    -- ========================================================================
    MAX(has_cutoff_configured) AS has_cutoff,
    AVG(cutoff_hours) AS avg_cutoff_hours,
    MAX(has_min_participant_requirement) AS has_min_participants,
    MAX(has_max_participant_limit) AS has_max_participants,
    MAX(has_capacity_limits) AS has_capacity_limits,
    
    -- VACANCY COVERAGE
    AVG(pct_dates_with_vacancy_info) AS avg_pct_dates_with_vacancy,
    MAX(has_available_slots) AS has_available_slots,
    
    -- METADATA
    COUNT(DISTINCT pricing_id) AS num_pricing_configs,
    MAX(config_snapshot_timestamp) AS latest_config_timestamp
    
  FROM production.supply_analytics.pricing_feature_audit_base
  GROUP BY tour_id, supplier_id
)

SELECT 
  -- IDENTIFIERS
  u.tour_id,
  u.supplier_id,
  
  -- SUPPLIER & TOUR ATTRIBUTES
  u.supplier_segment,
  u.is_managed,
  u.tour_title,
  u.location_name,
  u.activity_type,
  u.tour_category,
  u.date_of_creation,
  u.date_of_first_online,
  u.is_top_attraction,
  COALESCE(u.total_options, 0) AS total_options,
  
  -- ========================================================================
  -- INDIVIDUAL PRICING FEATURES - ALL CATEGORIES
  -- ========================================================================
  u.has_individual_pricing,
  u.has_adult_category,
  u.has_child_category,
  u.has_infant_category,
  u.has_youth_category,
  u.has_senior_category,
  u.has_student_category,
  u.has_military_category,
  u.has_eu_citizen_student_category,
  u.has_eu_citizen_category,
  u.num_individual_categories,
  u.num_individual_tiers,
  u.has_individual_scales,
  
  -- ========================================================================
  -- GROUP PRICING FEATURES
  -- ========================================================================
  u.has_group_pricing,
  u.num_group_configs,
  u.max_group_tiers,
  u.num_group_tiers,
  u.has_group_scales,
  u.has_additional_pax_for_group,
  
  -- ========================================================================
  -- ADD-ON FEATURES
  -- ========================================================================
  u.has_addons,
  u.num_addons,
  u.num_addon_tiers,
  u.has_addon_scales,
  
  -- ========================================================================
  -- PRICING SCALES (SEPARATED)
  -- ========================================================================
  u.has_native_scales,
  u.has_api_scales,
  u.has_scale_pricing,
  
  -- ========================================================================
  -- API PRICING (SEPARATED)
  -- ========================================================================
  u.has_static_api_pricing,
  u.has_live_dynamic_pricing,
  u.uses_api_pricing,
  
  -- ========================================================================
  -- OPERATIONAL FEATURES
  -- ========================================================================
  u.has_cutoff,
  u.avg_cutoff_hours,
  u.has_min_participants,
  u.has_max_participants,
  u.has_capacity_limits,
  u.avg_pct_dates_with_vacancy,
  u.has_available_slots,
  
  -- METADATA
  u.num_pricing_configs,
  u.latest_config_timestamp,
  
  -- ========================================================================
  -- PERFORMANCE METRICS (L3M)
  -- ========================================================================
  COALESCE(p.bookings_l3mo, 0) AS bookings_l3mo,
  COALESCE(p.gmv_l3mo, 0) AS gmv_l3mo,
  COALESCE(p.nr_l3mo, 0) AS nr_l3mo,
  COALESCE(p.tickets_l3mo, 0) AS tickets_l3mo,
  COALESCE(p.customers_l3mo, 0) AS customers_l3mo,
  
  -- PERFORMANCE METRICS (L6M)
  COALESCE(p.bookings_l6mo, 0) AS bookings_l6mo,
  COALESCE(p.gmv_l6mo, 0) AS gmv_l6mo,
  COALESCE(p.nr_l6mo, 0) AS nr_l6mo,
  COALESCE(p.tickets_l6mo, 0) AS tickets_l6mo,
  COALESCE(p.customers_l6mo, 0) AS customers_l6mo,
  
  -- PERFORMANCE METRICS (L12M)
  COALESCE(p.bookings_l12mo, 0) AS bookings_l12mo,
  COALESCE(p.gmv_l12mo, 0) AS gmv_l12mo,
  COALESCE(p.nr_l12mo, 0) AS nr_l12mo,
  COALESCE(p.tickets_l12mo, 0) AS tickets_l12mo,
  COALESCE(p.customers_l12mo, 0) AS customers_l12mo,
  
  -- PERFORMANCE METRICS (LIFETIME)
  COALESCE(p.bookings_lifetime, 0) AS bookings_lifetime,
  COALESCE(p.gmv_lifetime, 0) AS gmv_lifetime,
  COALESCE(p.nr_lifetime, 0) AS nr_lifetime,
  
  -- AVERAGE METRICS
  p.avg_booking_value_l12mo,
  p.avg_tickets_per_booking_l12mo,
  
  -- ENGAGEMENT
  p.last_booking_date,
  p.first_booking_date,
  p.days_since_last_booking,
  
  -- REPEAT CUSTOMER RATES
  p.repeat_customer_rate_l3mo,
  p.repeat_customer_rate_l6mo,
  p.repeat_customer_rate_l12mo,
  
  -- PERCENTILES & TIERS
  p.gmv_percentile,
  p.bookings_percentile,
  p.nr_percentile,
  p.basket_size_percentile,
  p.gmv_decile,
  p.gmv_quartile,
  p.gmv_tier,
  p.tour_value_tier,
  
  -- GROWTH
  p.booking_growth_trend_pct

FROM tour_level_usage u
LEFT JOIN production.supply_analytics.pricing_feature_audit_performance p
  ON u.tour_id = p.tour_id

ORDER BY u.tour_id;
